# Project 2: Optimization

In [ ]:
# FIXME: Enter your group name here
GROUP = "x"

## 2.0 Preparation

Ensure that you run this notebook in your project environment created for the previous project. *On Windows: ensure that you are using WSL!*

Install the `ax-platform` in your current environment using `pip`. Either use the console/bash or by running the following cell.

In [ ]:
!pip install ax-platform

Import necessary Python modules. We will re-use the FeniCS interface from the previous project. For that to work, you need to place this Jupyter notebook in the `project_fem` directory (the directory containing `main.py`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ax

from fem.interface.fenics_interface import FenicsInterface
from fem.base import grids

## 2.1 Create the blackbox-function 

Complete the function `solve_fem` below, which we will use for our optimization problem. The given signature and docstring of the function defines the inputs and outputs. Do not change the signature or docstring.

**(a) Implement the steps needed to solve the load-controlled elastoplastic cube.**

Use the `main.py` from the previous project as starting point (You can copy & paste most of it and modify only the necessary parts).

Apply Dirichlet (displacement) boundary conditions `bc_u`. Choose a bending deformation on the `"xn"` and `"xp"` boundaries. This results in a load-controlled bending problem. **Remove the unloading part.**

*Tip: disable plotting in the FenicsInterface `fi.plot = False` to accelerate computations.* 

**(b) Assemble the results dictionary.**

Assemble all frames in the `fe_histories` into numerical tensors (`np.asarray`) for each element. 

Use the `numpy` python module (`np.stack`, `np.reshape`). 

The final tensors should have the shape (T X Y F) with T: number of time frames, X and Y: grid dimensionality, F: number of components/features.

**(c) Compute additional features and metrics.**

Use the functions `np.min` and `np.max` and check out slicing in numpy.
add the metrics to the result dictionary

In [ ]:
# Do not change this cell!
#   This function is a useful helper to apply grid encoding to the frame data. The lines
#   using it later are already prepared for you. You do not need to use it anywhere else
#   in the notebook or modify this function.
def apply_grid_encoding(frame, grid_shape):
    """Grid encode the frame data."""
    Xi = frame["Xi"]
    origin = np.min(Xi, axis=0)
    geometry = tuple(np.max(Xi, axis=0) - np.min(Xi, axis=0))
    grid_coords = grids.generate_grid(geometry, grid_shape) + origin
    reduced_frame = {
        k: v
        for k, v in frame.items()
        if "fenics" not in k and "bc_" not in k and isinstance(v, np.ndarray)
    }
    del reduced_frame["Pi"]
    _, _, grid_encoded = grids.grid_encode(grid_coords, grid_shape, Xi, reduced_frame)
    return grid_encoded

In [ ]:
def solve_fem(E=7e10, u0=1e-3, nu=0.3, rho=0.0, sig0=2.50e8, Et=1e-9):
    """Solve the elastoplastic cube problem using the FenicsInterface.

    Args:
        E (float, optional): Young's (elastic) modulus. Defaults to 7e10.
        u0 (float, optional): Maximum displacement at the left and right edges of the cube. Defaults to 1e-3.
        nu (float, optional): Poisson's ratio. Defaults to 0.3.
        rho (float, optional): Density of the material. Defaults to 0.0.
        sig0 (float, optional): Initial yield stress. Defaults to 2.50e8.
        Et (float, optional): Tangent modulus for kinematic hardening. Defaults to 1e-9.

    Returns:
        dict: A dictionary containing the results of the simulation as feature tensors.
    """

    # FIXME Create a FenicsInterface object
    #   This is the main interface to the Fenics library
    #   It handles the setup of the problem, the mesh generation,
    #   the assembly of the system matrices and vectors,
    #   and the solution of the system



    fi = None



    # Define geometry, material properties, and boundary conditions
    #   Keep the geometry as a 2D square with length 1.0 in both x and y directions
    geometry = [1.0, 1.0]  # Length in x and y directions

    # Define material properties
    #    Bilinear elastoplastic material with kinematic hardening
    material = [E, nu, rho, sig0, Et]

    # Define properties
    #    Thickness of the material is 1.0, which is a common assumption for 2D problems
    thickness = 1.0  # Thickness of the material
    properties = [thickness]

    # FIXME Define boundary conditions
    #    At the left and right edges of the cube, we apply linear bending displacement.
    #    The dislacement are defined as a function of the y coordinate.
    #    The maximum displacement is defined as u0 as an input to the function.



    bc_u1 = dict()



    # Finite element settings
    grid_shape = [27, 27]  # Number of elements in x and y directions
    fi.interpolation_order = 2  # element interpolation order (1: linear, 2: quadratic)
    fi.min_increment = 5  # Minimum number of time increments
    fi.max_iteration = 100  # Maximum number of iterations
    fi.tol = 1e-10  # Tolerance for convergence
    fi.regularization = 1.0  # Regularization parameter

    fe_histories = []  # List to store the finite element histories

    # FIXME Load the geometry
    print("Loading geometry...")
    
    
    
    fe_history_load = None
    
    
    
    fe_histories += [fe_history_load]

    # Check if the results are numerically stable and correct
    assert (
        np.max(fi.extract_stress_values(fe_histories[0][-1]["Si"], stress_type="mises"))
        > 0.0
    )
    assert np.max(
        fi.extract_stress_values(fe_histories[0][-1]["Si"], stress_type="mises")
    ) <= sig0 * (1.0 + 1e-4)

    # FIXME Assemble the results into a dictionary containing the feature tensors
    #    The tensors should be of the following dimensions:
    #    T: Number of time frames (varies with the number of increments)
    #    X: Number of grid points in x direction (27 in this case)
    #    Y: Number of grid points in y direction (27 in this case)
    #    F: Number of features (2 for displacement and force, 6 for stress,
    #       1 for Von Mises stress)
    frames = [
        apply_grid_encoding(f, grid_shape)
        for h in fe_histories
        for f in h.values()
        if f
    ]
    frames = [
        {k: np.zeros_like(v) for k, v in frames[-1].items()}
    ] + frames  # add initial zero frame for initial state
    
    
    U = None
    F = None
    S = None
    Mises = None
    
    
    
    Mises = Mises[..., np.newaxis]  # Add a new axis for compatibility
    
    results = {
        "U": U,  # Displacement tensor of shape (T, X, Y, 2)
        "F": F,  # Force tensor of shape (T, X, Y, 2)
        "S": S,  # Stress tensor of shape (T, X, Y, 6)
        "Mises": Mises,  # Von Mises stress tensor of shape (T, X, Y, 1)
    }

    # FIXME Compute non-local and non-directional features from the results.
    #  Compute the magnitudes of the displacement and force vectors at each grid point
    #  for each time frame and store them in the results dictionary.
    
    
    
    u = None  # Magnitude of displacement tensor of shape (T, X, Y, 1)
    f = None  # Magnitude of force tensor of shape (T, X, Y, 1)
    results["max_u"] = None  # Max displacement magnitude tensor of shape (T, 1)
    results["min_u"] = None  # Min displacement magnitude tensor of shape (T, 1)
    results["max_f"] = None  # Max force magnitude tensor of shape (T, 1)
    results["min_f"] = None  # Min force magnitude tensor of shape (T, 1)
    results["max_mises"] = None  # Max Von Mises stress tensor of shape (T, 1)
    results["min_mises"] = None  # Min Von Mises stress tensor of shape (T, 1)
    

    # FIXME Compute metrics for the objective function
    #   The objective function is the maximum Von Mises stress over all time frames
    #   and the maximum displacement at the last time frame. Store the metrics in the
    #   results dictionary as scalars.
    
    
    
    results["objective_max_equivalent_stress"] = None
    results["objective_final_displacement"] = None
    
    
    
    return results

**(d) Try your implemented function.**

In [ ]:
results = solve_fem(E=7e10, u0=3e-3)

In [ ]:
for key, value in results.items():
    print(f"{key}: {value.shape if isinstance(value, np.ndarray) else value}")

**(e) Plot the results**

Use matplotlib to plot the von-Mises stress over the grid points for the last frame. Use `plt.imshow`.

In [ ]:
# FIXME Plot features over the grid
feature = "Mises"
vmin = np.min(results[feature])
vmax = np.max(results[feature])
plt.title(f"Group {GROUP}: {feature}")
plt.xlabel("X")
plt.ylabel("Y")




plt.colorbar()

Plot the load-displacement curve (`max_f` over `max_u`) using `plt.plot`. 

In [ ]:
# FIXME Plot the load-displacement curve
plt.title(f"Group {GROUP}: Load-displacement curve")



plt.xlabel("Max Displacement (m)")
plt.ylabel("Max Mises Stress (Pa)")

# 2.2 Solve an optimization problem using Ax

We will now use acive learning with Bayesian Optimization to investigate the physics-based model of the elastoplastic finite element code.

The function `solve_fem` that you implemented before will serve as blackbox function.

We will use the ax platform for this. Check out the tutorials and documetation on https://ax.dev/.
In particular, the Quickstart and Multi-Objective Optimization with Ax tutorials will be helpful.

**(a) Set up the experiment**

Vary the parameter `"E"` between $5e10$ and $9e10$ and the parameter `"u0"` between $1e-3$ and $5e-3$. Keep all other parameters in your blackbox functions at their default values. 

Implement a multi-objective optimization to maximize `"objective_max_equivalent_stress"` and minimize `"objective_final_displacement"`.

In [ ]:
client = ax.Client()
# FIXME Configure the experiment



# FIXME Configure the objective function




**(b) Run 20 trials of optimization**

Use the function `solve_fem` that you implemented before as blackbox function.

In [ ]:
for _ in range(20):
    # FIXME get the next trial, solve the FEM problem, extract the metrics, and 
    #   complete the trial
    
    
    trial_index = None
    parameters = {}
    
    
    print(f"Trial {trial_index}:")
    print(f"  Parameters: {parameters}")
    
    
    
    results = {}
    metrics = {}



    print(f"  Metrics: {metrics}")

**(c) Plot the analysis**

Observe the trade-off between the objectives and the influence of the parameters on the individual objectives in the contour plots (*Great to describe in your report!*).

In [ ]:
# FIXME Compute the analysis plots (check the Ax documentation for details)
